<a href="https://colab.research.google.com/github/cked007-glitch/IBVAP-Intelligent-Border-Surveillance/blob/main/FRS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 4.9 MB/s eta 0:00:00


In [6]:
import glob
import cv2
from ultralytics import YOLO

def auto_detect_and_process():
    # Automatically find any .mp4 file in the current directory
    video_files = glob.glob("*.mp4")

    if not video_files:
        print("[ERROR] No .mp4 video files found in the directory! Please upload a video.")
        return

    # Automatically pick the first available video file
    video_source = video_files[0]
    print(f"\n[AUTO-DETECTED] Found video file: '{video_source}'")

    model = YOLO("yolov8n.pt")
    cap = cv2.VideoCapture(video_source)

    if not cap.isOpened():
        print(f"[ERROR] Could not open video source: '{video_source}'.")
        return

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS) or 30)

    out = cv2.VideoWriter('universal_output.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    print(f"--- PROCESSING STREAM: {video_source} ---")
    frame_count = 0
    flagged_ids = set()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_count += 1

        # Removed 'vid_stride' as it does not apply to single frame arrays
        results = model.track(frame, classes=[0], conf=0.4, persist=True, verbose=False)

        if results[0].boxes.id is not None:
            for box, track_id in zip(results[0].boxes.xyxy, results[0].boxes.id):
                x1, y1, x2, y2 = [int(v) for v in box.tolist()]
                box_height = y2 - y1

                # Safer tensor extraction using .item()
                t_id = int(track_id.item())

                status = "TRACKED"
                color = (0, 255, 255)

                if box_height > 80:
                    status = "UNAUTHORIZED TARGET"
                    color = (0, 0, 255)

                    if t_id not in flagged_ids:
                        flagged_ids.add(t_id)
                        print(f"[SECURITY ALERT] Frame {frame_count} | Intruder ID {t_id} detected!")

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f"ID {t_id}: {status}", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        out.write(frame)

    cap.release()
    out.release()
    print(f"\n--- STREAM COMPLETE: Processed {frame_count} frames | Unique intruders flagged: {len(flagged_ids)} ---")
    print("Saved output as 'universal_output.mp4'.\n")

if __name__ == "__main__":
    auto_detect_and_process()


[AUTO-DETECTED] Found video file: 'designarena_video_jujdvdta.mp4'
--- PROCESSING STREAM: designarena_video_jujdvdta.mp4 ---
[SECURITY ALERT] Frame 16 | Intruder ID 2 detected!

--- STREAM COMPLETE: Processed 124 frames | Unique intruders flagged: 1 ---
Saved output as 'universal_output.mp4'.



In [5]:
from google.colab import files
files.download('universal_output.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>